In [27]:
%pip install pandas scikit-learn nltk
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

### Load Dataset

In [29]:
file_path = "data/raw/All_emotions.csv"

df = pd.read_csv(file_path)

print("shape:", df.shape)
df.head()

shape: (7261, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
1,dentist,It's important to keep the area clean.,neutral
2,patient,I try to brush twice a day like you said.,neutral
3,dentist,You might feel a bit of pressure.,neutral
4,patient,Should I open wider?,neutral


In [30]:
print(df.columns.tolist())

['speaker', 'utterance', 'emotion']


In [31]:
print(df["speaker"].unique())
print(df["speaker"].value_counts())

['patient' 'dentist']
speaker
patient    3636
dentist    3625
Name: count, dtype: int64


### Data Filtering

In [32]:
df["speaker"] = df["speaker"].astype(str).str.lower().str.strip()

df = df[df["speaker"] == "patient"]

print("After filtering only patient data:", df.shape)
df.head()

After filtering only patient data: (3636, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
2,patient,I try to brush twice a day like you said.,neutral
4,patient,Should I open wider?,neutral
6,patient,I am not paying for a mistake you made.,angry
8,patient,I can feel the cold water on my teeth.,neutral


### Data Cleaning

In [33]:
df = df[["utterance", "emotion"]].copy()
df.head()

,utterance,emotion
0,I can feel the cold water on my teeth.,neutral
2,I try to brush twice a day like you said.,neutral
4,Should I open wider?,neutral
6,I am not paying for a mistake you made.,angry
8,I can feel the cold water on my teeth.,neutral


In [34]:
df.dropna(subset=["utterance", "emotion"], inplace=True)

df["utterance"] = df["utterance"].astype(str)
df["emotion"] = df["emotion"].astype(str)

df = df[df["utterance"].str.strip() != ""]
df = df[df["emotion"].str.strip() != ""]

print("Shape after removing missing and empty values:", df.shape)

Shape after removing missing and empty values: (3636, 2)


### Text Cleaning

In [35]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["utterance"].apply(clean_text)

df = df[df["clean_text"].str.strip() != ""]

df[["utterance", "clean_text", "emotion"]].head()

,utterance,clean_text,emotion
0,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral
2,I try to brush twice a day like you said.,i try to brush twice a day like you said,neutral
4,Should I open wider?,should i open wider,neutral
6,I am not paying for a mistake you made.,i am not paying for a mistake you made,angry
8,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral


### Tokenization

In [36]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"
2,i try to brush twice a day like you said,"[i, try, to, brush, twice, a, day, like, you, ..."
4,should i open wider,"[should, i, open, wider]"
6,i am not paying for a mistake you made,"[i, am, not, paying, for, a, mistake, you, made]"
8,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"


### Data Preparation

In [37]:
X_text = df["clean_text"]
y = df["emotion"]

print("Number of samples:", len(X_text))
print("\nLabel distribution:")
print(y.value_counts())

Number of samples: 3636

Label distribution:
emotion
neutral    2856
disgust     247
fear        195
angry       156
sad          98
happy        84
Name: count, dtype: int64


### TF-IDF Vectorization

In [38]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_tfidf = vectorizer.fit_transform(X_text)

print("TF-IDF shape:", X_tfidf.shape)
print("\nตัวอย่าง feature 20 ตัวแรก:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF shape: (3636, 3000)

ตัวอย่าง feature 20 ตัวแรก:
['able' 'able to' 'about' 'about how' 'about it' 'about looking'
 'about losing' 'about my' 'about needing' 'about that' 'about the'
 'about this' 'about to' 'about two' 'about what' 'about year'
 'absolutely' 'absolutely revolting' 'accident' 'ache']


### TF-IDF Representation

In [39]:
tfidf_sample = pd.DataFrame(
    X_tfidf[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_sample.head()

,able,able to,about,about how,about it,about looking,about losing,about my,about needing,about that,...,you were,you will,you would,youll,youll see,your,your receptionist,your time,youre,youre right
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
import os

os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/cleaned_emotions_patient.csv", index=False, encoding="utf-8-sig")

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_df["emotion"] = y.values

tfidf_df.to_csv("data/processed/tfidf_features_patient.csv", index=False, encoding="utf-8-sig")

print("Files saved successfully:")
print("- cleaned_emotions_patient.csv")
print("- tfidf_features_patient.csv")

Files saved successfully:
- cleaned_emotions_patient.csv
- tfidf_features_patient.csv


### Train / Test Split

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Before SMOTE")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Before SMOTE
X_train shape: (2908, 3000)
X_test shape: (728, 3000)

Train label distribution:
emotion
neutral    2284
disgust     198
fear        156
angry       125
sad          78
happy        67
Name: count, dtype: int64

Test label distribution:
emotion
neutral    572
disgust     49
fear        39
angry       31
sad         20
happy       17
Name: count, dtype: int64


### Convert sparse matrix to dense

In [42]:
X_train_patient = X_train.toarray()
X_test_patient = X_test.toarray()

### Apply SMOTE on training set only

In [43]:
smote = SMOTE(random_state=42)

X_train_patient_sm, y_train_patient_sm = smote.fit_resample(
    X_train_patient, y_train
)

print("\nAfter SMOTE")
print("X_train_patient_sm shape:", X_train_patient_sm.shape)

print("\nTrain label distribution after SMOTE:")
print(pd.Series(y_train_patient_sm).value_counts())



After SMOTE
X_train_patient_sm shape: (13704, 3000)

Train label distribution after SMOTE:
emotion
neutral    2284
disgust    2284
sad        2284
happy      2284
fear       2284
angry      2284
Name: count, dtype: int64


###  Final ready data

In [44]:
X_train_final = X_train_patient_sm
y_train_final = y_train_patient_sm
X_test_final = X_test_patient
y_test_final = y_test

print("\nFinal data ready for model")
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)
print("X_test_final:", X_test_final.shape)
print("y_test_final:", y_test_final.shape)



Final data ready for model
X_train_final: (13704, 3000)
y_train_final: (13704,)
X_test_final: (728, 3000)
y_test_final: (728,)


### Save train/test files

In [45]:
feature_names = vectorizer.get_feature_names_out()

train_smote_df = pd.DataFrame(X_train_final, columns=feature_names)
train_smote_df["emotion"] = y_train_final.values

test_df = pd.DataFrame(X_test_final, columns=feature_names)
test_df["emotion"] = y_test_final.values

train_smote_df.to_csv("data/processed/patient_train_tfidf_smote.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("data/processed/patient_test_tfidf.csv", index=False, encoding="utf-8-sig")

print("\nFiles saved successfully:")
print("- patient_train_tfidf_smote.csv")
print("- patient_test_tfidf.csv")


Files saved successfully:
- patient_train_tfidf_smote.csv
- patient_test_tfidf.csv


### Import Model

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pandas as pd

In [47]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n===== {model_name} =====")
    print(f"Accuracy      : {acc:.4f}")
    print(f"F1-score Macro: {f1_macro:.4f}")
    print(f"F1-score Weighted: {f1_weighted:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "Confusion_Matrix": cm
    }

### Logistic Regression

In [48]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

lr_result = evaluate_model(
    lr_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Logistic Regression"
)


===== Logistic Regression =====
Accuracy      : 0.9093
F1-score Macro: 0.8494
F1-score Weighted: 0.9108

Confusion Matrix:
[[ 31   0   0   0   0   0]
 [  0  41   1   0   7   0]
 [  0   5  17   0  17   0]
 [  0   0   0  17   0   0]
 [  0  12  22   0 536   2]
 [  0   0   0   0   0  20]]

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        31
     disgust       0.71      0.84      0.77        49
        fear       0.42      0.44      0.43        39
       happy       1.00      1.00      1.00        17
     neutral       0.96      0.94      0.95       572
         sad       0.91      1.00      0.95        20

    accuracy                           0.91       728
   macro avg       0.83      0.87      0.85       728
weighted avg       0.91      0.91      0.91       728



### Support Vector Machine

In [49]:
svm_model = SVC(kernel='linear', random_state=42)

svm_result = evaluate_model(
    svm_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Support Vector Machine"
)


===== Support Vector Machine =====
Accuracy      : 0.9231
F1-score Macro: 0.8685
F1-score Weighted: 0.9212

Confusion Matrix:
[[ 31   0   0   0   0   0]
 [  0  37   0   0  12   0]
 [  0   2  18   0  19   0]
 [  0   0   0  17   0   0]
 [  0   5  17   0 549   1]
 [  0   0   0   0   0  20]]

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        31
     disgust       0.84      0.76      0.80        49
        fear       0.51      0.46      0.49        39
       happy       1.00      1.00      1.00        17
     neutral       0.95      0.96      0.95       572
         sad       0.95      1.00      0.98        20

    accuracy                           0.92       728
   macro avg       0.88      0.86      0.87       728
weighted avg       0.92      0.92      0.92       728



### Naive Bayes

In [50]:
nb_model = MultinomialNB()

nb_result = evaluate_model(
    nb_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Naive Bayes"
)


===== Naive Bayes =====
Accuracy      : 0.7775
F1-score Macro: 0.6959
F1-score Weighted: 0.8066

Confusion Matrix:
[[ 31   0   0   0   0   0]
 [  0  45   2   0   2   0]
 [  3   9  22   1   4   0]
 [  0   0   0  17   0   0]
 [  1  47  65  12 431  16]
 [  0   0   0   0   0  20]]

Classification Report:
              precision    recall  f1-score   support

       angry       0.89      1.00      0.94        31
     disgust       0.45      0.92      0.60        49
        fear       0.25      0.56      0.34        39
       happy       0.57      1.00      0.72        17
     neutral       0.99      0.75      0.85       572
         sad       0.56      1.00      0.71        20

    accuracy                           0.78       728
   macro avg       0.61      0.87      0.70       728
weighted avg       0.88      0.78      0.81       728



### Random Forest

In [51]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_result = evaluate_model(
    rf_model,
    X_train_final,
    y_train_final,
    X_test_final,
    y_test_final,
    "Random Forest"
)


===== Random Forest =====
Accuracy      : 0.9038
F1-score Macro: 0.7804
F1-score Weighted: 0.8840

Confusion Matrix:
[[ 31   0   0   0   0   0]
 [  0  24   1   0  24   0]
 [  0   2   4   0  33   0]
 [  0   0   0  17   0   0]
 [  0   4   5   0 563   0]
 [  0   1   0   0   0  19]]

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        31
     disgust       0.77      0.49      0.60        49
        fear       0.40      0.10      0.16        39
       happy       1.00      1.00      1.00        17
     neutral       0.91      0.98      0.94       572
         sad       1.00      0.95      0.97        20

    accuracy                           0.90       728
   macro avg       0.85      0.75      0.78       728
weighted avg       0.88      0.90      0.88       728



### Result

In [52]:
results_df = pd.DataFrame([
    {
        "Model": lr_result["Model"],
        "Accuracy": lr_result["Accuracy"],
        "F1_Macro": lr_result["F1_Macro"],
        "F1_Weighted": lr_result["F1_Weighted"]
    },
    {
        "Model": svm_result["Model"],
        "Accuracy": svm_result["Accuracy"],
        "F1_Macro": svm_result["F1_Macro"],
        "F1_Weighted": svm_result["F1_Weighted"]
    },
    {
        "Model": nb_result["Model"],
        "Accuracy": nb_result["Accuracy"],
        "F1_Macro": nb_result["F1_Macro"],
        "F1_Weighted": nb_result["F1_Weighted"]
    },
    {
        "Model": rf_result["Model"],
        "Accuracy": rf_result["Accuracy"],
        "F1_Macro": rf_result["F1_Macro"],
        "F1_Weighted": rf_result["F1_Weighted"]
    }
])

results_df = results_df.sort_values(by="F1_Macro", ascending=False)
results_df

,Model,Accuracy,F1_Macro,F1_Weighted
1,Support Vector Machine,0.923077,0.868487,0.921239
0,Logistic Regression,0.909341,0.849352,0.910805
3,Random Forest,0.903846,0.780376,0.884043
2,Naive Bayes,0.777473,0.695858,0.806562
